In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from scipy import stats
from scipy.stats import normaltest, skew, kurtosis, mannwhitneyu, kruskal, spearmanr
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report
from sklearn.decomposition import PCA
import joblib
import json
from datetime import datetime

# buat OCR (kalau available)
try:
    import pytesseract
    pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    OCR_AVAILABLE = True
except:
    OCR_AVAILABLE = False

warnings.filterwarnings('ignore')

# Set path
MODEL_DIR = r'D:\Project\disdukcapil_project\model'
DATASET_DIR = os.path.join(MODEL_DIR, 'dataset')
TRAIN_DIR = os.path.join(DATASET_DIR, 'Train')

# Set style buat visualisasi
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('='*70)
print('PROYEK DATA SCIENCE: ANALISIS KUALITAS GAMBAR OCR')
print('='*70)
print(f'\nFolder Dataset: {TRAIN_DIR}')
print(f'Tesseract Available: {OCR_AVAILABLE}')
print(f'Total Files: {len(list(Path(TRAIN_DIR).glob("*.png")))} gambar')

In [ ]:
# Tampilkan daftar fitur yang akan diekstrak
fitur_dict = {
    'No.': [],
    'Kategori': [],
    'Nama Fitur': [],
    'Deskripsi': [],
    'Satuan': []
}

daftar_fitur = [
    (1, 'Dimensi', 'height', 'Tinggi gambar dalam piksel', 'piksel'),
    (2, 'Dimensi', 'width', 'Lebar gambar dalam piksel', 'piksel'),
    (3, 'Dimensi', 'aspect_ratio', 'Rasio lebar terhadap tinggi', 'rasio'),
    (4, 'Dimensi', 'pixel_count', 'Total jumlah piksel', 'jumlah'),
    (5, 'Kecerahan', 'mean_brightness', 'Rata-rata kecerahan piksel', '0-255'),
    (6, 'Kecerahan', 'std_brightness', 'Standar deviasi kecerahan', '0-255'),
    (7, 'Kecerahan', 'median_brightness', 'Nilai median kecerahan', '0-255'),
    (8, 'Kecerahan', 'min_brightness', 'Nilai kecerahan minimum', '0-255'),
    (9, 'Kecerahan', 'max_brightness', 'Nilai kecerahan maksimum', '0-255'),
    (10, 'Kualitas', 'sharpness', 'Varians Laplacian (ketajaman)', 'varians'),
    (11, 'Kualitas', 'contrast', 'Kontras gambar', 'standar deviasi'),
    (12, 'Kualitas', 'edge_density', 'Kepadatan tepi gambar', 'rasio'),
    (13, 'Kualitas', 'avg_gradient', 'Rata-rata gradien', 'nilai'),
    (14, 'Tekstur', 'texture_variance', 'Varians tekstur lokal', 'varians'),
    (15, 'Noise', 'noise_level', 'Tingkat noise gambar', 'standar deviasi'),
    (16, 'Histogram', 'entropy', 'Entropi histogram', 'bit'),
    (17, 'Histogram', 'hist_skew', 'Skewness histogram', 'nilai'),
    (18, 'Histogram', 'hist_kurtosis', 'Kurtosis histogram', 'nilai'),
    (19, 'OCR', 'ocr_confidence', 'Confidence Tesseract OCR', 'persen'),
    (20, 'OCR', 'text_length', 'Panjang teks hasil OCR', 'karakter'),
    (21, 'OCR', 'word_count', 'Jumlah kata hasil OCR', 'jumlah'),
]

for item in daftar_fitur:
    fitur_dict['No.'].append(item[0])
    fitur_dict['Kategori'].append(item[1])
    fitur_dict['Nama Fitur'].append(item[2])
    fitur_dict['Deskripsi'].append(item[3])
    fitur_dict['Satuan'].append(item[4])

df_fitur = pd.DataFrame(fitur_dict)
print('\nDaftar Fitur yang Diekstrak:')
print('='*80)
print(df_fitur.to_string(index=False))
print('='*80)
print(f'\nTotal Fitur: {len(daftar_fitur)} fitur')

In [ ]:
def extract_fitur_gambar(image_path):
    """
    Extract semua fitur dari satu gambar
    
    Args:
        image_path: Path ke file gambar
    
    Returns:
        dict: Semua fitur yang diekstrak
    """
    # Load gambar
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    # Convert ke grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    
    # === 1. FITUR DIMENSI ===
    aspect_ratio = width / height
    pixel_count = height * width
    
    # === 2. FITUR BRIGHTNESS (KECERAHAN) ===
    mean_brightness = np.mean(gray)
    std_brightness = np.std(gray)
    median_brightness = np.median(gray)
    min_brightness = np.min(gray)
    max_brightness = np.max(gray)
    
    # === 3. FITUR KUALITAS ===
    # Sharpness (ketajaman) - Varians Laplacian
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    sharpness = laplacian.var()
    
    # Contrast
    contrast = std_brightness
    
    # Edge Density (kepadatan tepi)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / (height * width)
    
    # Average Gradient
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    gradient_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    avg_gradient = np.mean(gradient_magnitude)
    
    # === 4. FITUR TEKSTUR ===
    # Texture Variance - variasi di neighborhood lokal
    kernel = np.ones((5, 5), np.float32) / 25
    local_mean = cv2.filter2D(gray.astype(np.float32), -1, kernel)
    local_variance = cv2.filter2D((gray.astype(np.float32) - local_mean)**2, -1, kernel)
    texture_variance = np.mean(local_variance)
    
    # === 5. FITUR NOISE ===
    # Noise level - standar deviasi dari high-frequency components
    high_freq = cv2.Laplacian(gray, cv2.CV_64F)
    noise_level = np.std(high_freq)
    
    # === 6. FITUR HISTOGRAM ===
    # Histogram entropy
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = hist.flatten() / hist.sum()  # Normalize
    
    # Entropy
    hist_nonzero = hist[hist > 0]
    entropy = -np.sum(hist_nonzero * np.log2(hist_nonzero))
    
    # Skewness & Kurtosis
    from scipy.stats import skew as skew_func, kurtosis as kurtosis_func
    hist_skew = skew_func(gray.flatten())
    hist_kurtosis = kurtosis_func(gray.flatten())
    
    # === 7. FITUR OCR (jika available) ===
    ocr_confidence = 0
    text_length = 0
    word_count = 0
    
    if OCR_AVAILABLE:
        try:
            # Preprocess untuk OCR
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
            enhanced = clahe.apply(gray)
            
            # Get OCR data
            data = pytesseract.image_to_data(enhanced, lang='ind', 
                                              config='--psm 6', 
                                              output_type=pytesseract.Output.DICT)
            
            # Confidence scores
            confidences = [int(c) for c in data['conf'] 
                          if str(c).isdigit() and int(c) > 0]
            ocr_confidence = np.mean(confidences) if confidences else 0
            
            # Text statistics
            text = pytesseract.image_to_string(enhanced, lang='ind', config='--psm 6')
            text_length = len(text.strip())
            word_count = len(text.split())
            
        except Exception as e:
            pass
    
    # Compile semua fitur
    fitur = {
        # Dimensi
        'height': height,
        'width': width,
        'aspect_ratio': aspect_ratio,
        'pixel_count': pixel_count,
        
        # Brightness
        'mean_brightness': mean_brightness,
        'std_brightness': std_brightness,
        'median_brightness': median_brightness,
        'min_brightness': min_brightness,
        'max_brightness': max_brightness,
        
        # Kualitas
        'sharpness': sharpness,
        'contrast': contrast,
        'edge_density': edge_density,
        'avg_gradient': avg_gradient,
        'texture_variance': texture_variance,
        'noise_level': noise_level,
        
        # Histogram
        'entropy': entropy,
        'hist_skew': hist_skew,
        'hist_kurtosis': hist_kurtosis,
        
        # OCR
        'ocr_confidence': ocr_confidence,
        'text_length': text_length,
        'word_count': word_count
    }
    
    return fitur, img, gray

print('Fungsi extract fitur sudah siap!')

In [ ]:
# Load semua gambar dari dataset
print('='*70)
print('PROSES EKSTRAKSI FITUR DARI SEMUA GAMBAR')
print('='*70)

image_files = list(Path(TRAIN_DIR).glob('*.png')) + list(Path(TRAIN_DIR).glob('*.jpg'))
print(f'\nDitemukan {len(image_files)} gambar di {TRAIN_DIR}')

# Extract fitur dari semua gambar
all_fitur = []
image_names = []
failed_images = []

print('\nMemproses gambar...')
print('-'*70)

for i, img_path in enumerate(image_files):
    try:
        fitur, img, gray = extract_fitur_gambar(str(img_path))
        if fitur is not None:
            all_fitur.append(fitur)
            image_names.append(img_path.name)
            print(f'{i+1:2d}. {img_path.name}: OK ({len(fitur)} fitur)')
        else:
            failed_images.append(img_path.name)
            print(f'{i+1:2d}. {img_path.name}: GAGAL')
    except Exception as e:
        print(f'{i+1:2d}. {img_path.name}: ERROR - {e}')
        failed_images.append(img_path.name)

print('-'*70)
print(f'\nRingkasan:')
print(f'  Berhasil: {len(all_fitur)} gambar')
print(f'  Gagal: {len(failed_images)} gambar')
print(f'  Total fitur per gambar: {len(all_fitur[0]) if all_fitur else 0} fitur')

In [ ]:
# Buat DataFrame dari fitur yang diekstrak
df = pd.DataFrame(all_fitur)
df['image_name'] = image_names

print('\n' + '='*70)
print('STATISTIK DESKRIPTIF DATASET')
print('='*70)
print(f'\nShape Dataset: {df.shape}')
print(f'  - Baris (observasi): {df.shape[0]}')
print(f'  - Kolom (fitur): {df.shape[1]}')

print('\nDaftar Kolom:')
print(df.columns.tolist())

print('\n' + '-'*70)
print('Statistik Deskriptif:')
print('-'*70)
print(df.describe().round(2))

In [ ]:
# VISUALISASI 1: Distribusi Fitur Kualitas Utama
# Alasan: Melihat sebaran kualitas gambar dalam dataset

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribusi Fitur Kualitas Gambar', fontsize=16, fontweight='bold')

# 1. Mean Brightness
axes[0, 0].hist(df['mean_brightness'], bins=15, color='skyblue', 
               edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['mean_brightness'].mean(), color='red', 
               linestyle='--', linewidth=2, label=f'Mean: {df["mean_brightness"].mean():.1f}')
axes[0, 0].set_title('Distribusi Kecerahan Rata-rata', fontweight='bold')
axes[0, 0].set_xlabel('Kecerahan (0-255)')
axes[0, 0].set_ylabel('Frekuensi')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Sharpness
axes[0, 1].hist(df['sharpness'], bins=15, color='lightcoral', 
               edgecolor='black', alpha=0.7)
axes[0, 1].axvline(df['sharpness'].mean(), color='red', 
               linestyle='--', linewidth=2, label=f'Mean: {df["sharpness"].mean():.1f}')
axes[0, 1].set_title('Distribusi Ketajaman (Sharpness)', fontweight='bold')
axes[0, 1].set_xlabel('Sharpness (Varians Laplacian)')
axes[0, 1].set_ylabel('Frekuensi')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Contrast
axes[0, 2].hist(df['contrast'], bins=15, color='lightgreen', 
               edgecolor='black', alpha=0.7)
axes[0, 2].axvline(df['contrast'].mean(), color='red', 
               linestyle='--', linewidth=2, label=f'Mean: {df["contrast"].mean():.1f}')
axes[0, 2].set_title('Distribusi Kontras', fontweight='bold')
axes[0, 2].set_xlabel('Kontras (Std Dev)')
axes[0, 2].set_ylabel('Frekuensi')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. Edge Density
axes[1, 0].hist(df['edge_density'], bins=15, color='plum', 
               edgecolor='black', alpha=0.7)
axes[1, 0].axvline(df['edge_density'].mean(), color='red', 
               linestyle='--', linewidth=2, label=f'Mean: {df["edge_density"].mean():.3f}')
axes[1, 0].set_title('Distribusi Kepadatan Tepi', fontweight='bold')
axes[1, 0].set_xlabel('Edge Density')
axes[1, 0].set_ylabel('Frekuensi')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 5. Noise Level
axes[1, 1].hist(df['noise_level'], bins=15, color='lightpink', 
               edgecolor='black', alpha=0.7)
axes[1, 1].axvline(df['noise_level'].mean(), color='red', 
               linestyle='--', linewidth=2, label=f'Mean: {df["noise_level"].mean():.1f}')
axes[1, 1].set_title('Distribusi Tingkat Noise', fontweight='bold')
axes[1, 1].set_xlabel('Noise Level')
axes[1, 1].set_ylabel('Frekuensi')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. OCR Confidence
axes[1, 2].hist(df['ocr_confidence'], bins=15, color='khaki', 
               edgecolor='black', alpha=0.7)
axes[1, 2].axvline(df['ocr_confidence'].mean(), color='red', 
               linestyle='--', linewidth=2, label=f'Mean: {df["ocr_confidence"].mean():.1f}')
axes[1, 2].set_title('Distribusi Confidence OCR', fontweight='bold')
axes[1, 2].set_xlabel('OCR Confidence (%)')
axes[1, 2].set_ylabel('Frekuensi')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'distribusi_fitur.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*70)
print('INTERPRETASI VISUALISASI 1: DISTRIBUSI FITUR KUALITAS')
print('='*70)
print('\n1. KECERAHAN (Mean Brightness):')
print(f'   - Rata-rata: {df["mean_brightness"].mean():.1f}')
print(f'   - Rentang: {df["mean_brightness"].min():.0f} - {df["mean_brightness"].max():.0f}')
print(f'   - Interpretasi: Kecerahan yang bagus buat OCR itu sekitar 100-180.')
if df['mean_brightness'].mean() < 100:
    print(f'   - Dataset terlalu GELAP, perlu pencahayaan lebih')
elif df['mean_brightness'].mean() > 180:
    print(f'   - Dataset terlalu TERANG, bisa menyebabkan silau')
else:
    print(f'   - Kecerahan NORMAL, cocok buat OCR')

print('\n2. KETAJAMAN (Sharpness):')
print(f'   - Rata-rata: {df["sharpness"].mean():.1f}')
print(f'   - Interpretasi: Sharpness > 100 = gambar JERNIH, < 50 = gambar BLUR')
blur_count = (df['sharpness'] < 50).sum()
print(f'   - Gambar blur: {blur_count} ({blur_count/len(df)*100:.1f}%)')
print(f'   - Rekomendasi: Gambar blur perlu diperbaiki/dibuang')

print('\n3. KONTRAS:')
print(f'   - Rata-rata: {df["contrast"].mean():.1f}')
print(f'   - Interpretasi: Kontras tinggi = teks lebih jelas')

print('\n4. NOISE LEVEL:')
print(f'   - Rata-rata: {df["noise_level"].mean():.1f}')
print(f'   - Interpretasi: Noise tinggi bisa mengganggu pembacaan teks')

print('\n5. OCR CONFIDENCE:')
print(f'   - Rata-rata: {df["ocr_confidence"].mean():.1f}%')
print(f'   - Interpretasi: Confidence > 70% = OCR bekerja dengan baik')
low_ocr = (df['ocr_confidence'] < 50).sum()
print(f'   - Gambar dengan confidence rendah: {low_ocr} ({low_ocr/len(df)*100:.1f}%)')

In [ ]:
# VISUALISASI 2: Correlation Heatmap
# Alasan: Melihat hubungan antar-fitur, fitur yang berkorelasi tinggi bisa redundant

# Pilih kolom numerik saja
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()

# Plot heatmap
plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, mask=mask,
            cbar_kws={'label': 'Korelasi'})
plt.title('Matriks Korelasi Antar Fitur', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*70)
print('INTERPRETASI VISUALISASI 2: CORRELATION HEATMAP')
print('='*70)
print('\nKorelasi Tinggi (> 0.7 atau < -0.7):')

# Cari korelasi tinggi
high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr.append({
                'Fitur 1': correlation_matrix.columns[i],
                'Fitur 2': correlation_matrix.columns[j],
                'Korelasi': round(corr_val, 3)
            })

if high_corr:
    df_corr = pd.DataFrame(high_corr)
    print(df_corr.to_string(index=False))
    print('\nInterpretasi:')
    print('- Fitur yang berkorelasi tinggi mungkin redundant')
    print('- Pertimbangkan untuk menghapus salah satu saat modelling')
else:
    print('Tidak ada korelasi yang sangat tinggi antar-fitur')
    print('Semua fitur cukup independen, bagus buat modelling')

In [ ]:
# VISUALISASI 3: Box Plot untuk Melihat Outliers
# Alasan: Mendeteksi outlier yang bisa mempengaruhi model

# Pilih fitur penting buat box plot
fitur_boxplot = ['mean_brightness', 'sharpness', 'contrast', 'edge_density', 
                'noise_level', 'ocr_confidence']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Box Plot Fitur Kualitas (Deteksi Outlier)', fontsize=16, fontweight='bold')

for idx, fitur in enumerate(fitur_boxplot):
    row, col = idx // 3, idx % 3
    
    axes[row, col].boxplot(df[fitur], vert=True, patch_artist=True,
                            boxprops=dict(facecolor='lightblue', alpha=0.7),
                            medianprops=dict(color='red', linewidth=2),
                            whiskerprops=dict(color='black', linewidth=1.5))
    
    # Tambahkan info statistik
    median_val = df[fitur].median()
    q1 = df[fitur].quantile(0.25)
    q3 = df[fitur].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df[(df[fitur] < lower_bound) | (df[fitur] > upper_bound)][fitur]
    
    axes[row, col].set_title(f'{fitur.replace("_", " ").title()}', fontweight='bold')
    axes[row, col].set_ylabel('Nilai')
    axes[row, col].grid(True, alpha=0.3)
    
    # Tambahkan teks info outlier
    info_text = f'Outlier: {len(outliers)}'
    axes[row, col].text(0.95, 0.95, info_text, transform=axes[row, col].transAxes,
                     fontsize=9, verticalalignment='top', horizontalalignment='right',
                     bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'boxplot_outliers.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*70)
print('INTERPRETASI VISUALISASI 3: BOX PLOT (OUTLIER DETECTION)')
print('='*70)

for fitur in fitur_boxplot:
    q1 = df[fitur].quantile(0.25)
    q3 = df[fitur].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df[(df[fitur] < lower_bound) | (df[fitur] > upper_bound)][fitur]
    
    print(f'\n{fitur.upper().replace("_", " ")}:')
    print(f'  - Q1: {q1:.2f}')
    print(f'  - Q3: {q3:.2f}')
    print(f'  - IQR: {iqr:.2f}')
    print(f'  - Batas Bawah: {lower_bound:.2f}')
    print(f'  - Batas Atas: {upper_bound:.2f}')
    print(f'  - Jumlah Outlier: {len(outliers)}')
    
    if len(outliers) > 0:
        print(f'  - Nilai outlier: {outliers.tolist()}')
        print(f'  - Rekomendasi: Pertimbangkan untuk {"handling" if len(outliers) < 3 else "menghapus"} outlier')
    else:
        print(f'  - Tidak ada outlier, data tersebar normal')

In [ ]:
# === TEKNIK 1: HANDLING MISSING VALUES ===
print('='*70)
print('TEKNIK 1: HANDLING MISSING VALUES')
print('='*70)

# Cek missing values
print('\nCek Missing Values Sebelum:')
print(df.isnull().sum())

# Jika ada missing values di ocr_confidence (kemungkinan OCR gagal)
if df['ocr_confidence'].isnull().sum() > 0:
    print(f'\nTerdapat {df["ocr_confidence"].isnull().sum()} missing values di ocr_confidence')
    print('Menangani dengan mean imputation...')
    mean_conf = df['ocr_confidence'].mean()
    df['ocr_confidence'].fillna(mean_conf, inplace=True)
    print('Missing values sudah diisi dengan rata-rata')

# Teknik yang diterapkan: Mean Imputation
# Alasan: Mengisi missing value dengan rata-rata kolom tersebut
# Kelebihan: Mudah dan cepat
# Kekurangan: Bisa bias kalau ada banyak outlier

print('\nCek Missing Values Sesudah:')
print(df.isnull().sum())
print('\nTidak ada missing values lagi, dataset siap diproses!')

In [ ]:
# === TEKNIK 2: HANDLING OUTLIERS (WINSORIZATION) ===
print('='*70)
print('TEKNIK 2: HANDLING OUTLIERS - WINSORIZATION')
print('='*70)

# Simpan dataframe asli buat perbandingan
df_original = df.copy()

# Fungsi Winsorization
def winsorize_series(series, lower_percentile=5, upper_percentile=95):
    """
    Winsorization: Capping outlier pada percentile tertentu
    """
    lower_limit = np.percentile(series, lower_percentile)
    upper_limit = np.percentile(series, upper_percentile)
    return series.clip(lower=lower_limit, upper=upper_limit)

# Terapkan pada fitur numerik (kecuali target/label)
fitur_numerik = df.select_dtypes(include=[np.number]).columns.tolist()
fitur_numerik = [f for f in fitur_numerik if f not in ['height', 'width', 'pixel_count']]

print(f'\nFitur yang di-winsorize: {len(fitur_numerik)} fitur')

# Statistik sebelum winsorization
print('\nStatistik Sebelum Winsorization:')
print(df[fitur_numerik].describe().round(2))

# Terapkan Winsorization
for fitur in fitur_numerik:
    df[fitur] = winsorize_series(df[fitur])

# Statistik sesudah winsorization
print('\n' + '-'*70)
print('Statistik Sesudah Winsorization:')
print(df[fitur_numerik].describe().round(2))

print('\n' + '-'*70)
print('Interpretasi Winsorization:')
print('-'70)
print('\nTeknik yang diterapkan: Winsorization (Capping at 5th and 95th percentile)')
print('\nAlasan Pemilihan:')
print('- Menjaga distribusi data tetap natural')
print('- Hanya membatasi extreme values, bukan menghapusnya')
print('- Lebih konservatif dibanding trimming')

print('\nKelebihan:')
print('- Outlier tidak hilang, hanya dibatasi')
print('- Tidak mengurangi ukuran sample')

print('\nKekurangan:')
print('- Bisa mempengaruhi statistik jika outlier terlalu extreme')
print('- Perlu penyesuaian percentile tergantung distribusi data')

In [ ]:
# === TEKNIK 3: FEATURE SCALING (STANDARDIZATION) ===
print('='*70)
print('TEKNIK 3: FEATURE SCALING - STANDARDIZATION')
print('='*70)

# Pilih fitur yang akan di-scale
fitur_scaling = ['mean_brightness', 'std_brightness', 'sharpness', 'contrast',
               'edge_density', 'avg_gradient', 'texture_variance', 'noise_level',
               'entropy', 'hist_skew', 'hist_kurtosis']

# Simpan data sebelum scaling
df_before_scaling = df[fitur_scaling].copy()

# Terapkan Standardization (Z-score normalization)
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[fitur_scaling] = scaler.fit_transform(df[fitur_scaling])

# Visualisasi sebelum dan sesudah scaling
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Efek Standardization pada Distribusi Fitur', fontsize=14, fontweight='bold')

# Tampilkan 4 fitur sebagai contoh
fitur_contoh = fitur_scaling[:4]

for idx, fitur in enumerate(fitur_contoh):
    # Before scaling
    axes[0, idx].hist(df_before_scaling[fitur], bins=15, color='lightblue', 
                      edgecolor='black', alpha=0.7)
    axes[0, idx].set_title(f'Before: {fitur}', fontweight='bold')
    axes[0, idx].set_ylabel('Frekuensi')
    axes[0, idx].grid(True, alpha=0.3)
    
    # After scaling
    axes[1, idx].hist(df_scaled[fitur], bins=15, color='lightgreen', 
                      edgecolor='black', alpha=0.7)
    axes[1, idx].set_title(f'After: {fitur}', fontweight='bold')
    axes[1, idx].set_ylabel('Frekuensi')
    axes[1, idx].axvline(0, color='red', linestyle='--', label='Mean=0')
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'effect_of_scaling.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '-'*70)
print('Interpretasi Feature Scaling:')
print('-'70)
print('\nTeknik yang diterapkan: Standardization (Z-score normalization')
print('\nRumus: z = (x - μ) / σ')
print('  where: x = nilai asli, μ = mean, σ = standard deviation')

print('\nAlasan Pemilihan Standardization (bukan Normalization):')
print('- Menjaga outlier tetap terdeteksi (tidak dikompres ke 0-1)')
print('- Cocok buat data yang diasumsikan berdistribusi normal')
print('- Dibutuhkan oleh banyak algoritma ML (SVM, Logistic Regression, Neural Networks)')

print('\nKelebihan:')
print('- Mean = 0, Std Dev = 1 (memudahkan komparasi antar fitur)')
print('- Mendekati distribusi normal')

print('\nKapan pakai Standardization vs Normalization:')
print('- Standardization: Data ada outlier, distribusi normal')
print('- Normalization (Min-Max): Data bounded, ga ada outlier, neural networks')

In [ ]:
# === TEKNIK 4: FEATURE SELECTION (PCA) ===
print('='*70)
print('TEKNIK 4: FEATURE SELECTION - PCA (Principal Component Analysis)')
print('='*70)

# Siapkan data buat PCA
X_for_pca = df_scaled[fitur_scaling].values

# Terapkan PCA
pca = PCA()
X_pca = pca.fit_transform(X_for_pca)

# Scree plot - Variance explained
plt.figure(figsize=(12, 6))
plt.bar(range(1, len(pca.explained_variance_ratio_)+1), 
        pca.explained_variance_ratio_, alpha=0.5, align='center',
        label='Individual Variance')
plt.step(range(1, len(pca.explained_variance_ratio_)+1), 
         np.cumsum(pca.explained_variance_ratio_), where='mid',
         label='Cumulative Variance')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance')
plt.xlabel('Principal Component')
plt.ylabel('Proportion of Variance Explained')
plt.title('Scree Plot - PCA Variance Explained', fontweight='bold')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(MODEL_DIR, 'pca_scree_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

# Tampilkan explained variance ratio
print('\nExplained Variance Ratio per PC:')
print('-'*70)
for i in range(min(10, len(pca.explained_variance_ratio_))):
    cumsum = np.sum(pca.explained_variance_ratio_[:i+1])
    print(f'PC{i+1:2d}: {pca.explained_variance_ratio_[i]*100:5.2f}% | Cumulative: {cumsum*100:5.2f}%')

print('\n' + '-'*70)
print('Interpretasi PCA:')
print('-'70)
print('\nTeknik yang diterapkan: Principal Component Analysis (PCA)')
print('\nAlasan Pemilihan:')
print('- Mengurangi dimensi (dimensionality reduction)')
print('- Menghilangkan multicollinearity antar-fitur')
print('- Mempercepat training model')

# Cari jumlah PC buat 95% variance
n_components_95 = np.argmax(np.cumsum(pca.explained_variance_ratio_) >= 0.95) + 1

print(f'\nKomponen buat 95% variance: {n_components_95} PC')
print(f'Komponen asli: {len(fitur_scaling)} fitur')
print(f'Pengurangan dimensi: {len(fitur_scaling) - n_components_95} fitur')
print(f'Persentase pengurangan: {(1 - n_components_95/len(fitur_scaling))*100:.1f}%')

print('\nKelebihan:')
print('- Mengurangi kompleksitas model')
print('- Mempercepat training')
print('- Mengurangi overfitting')

print('\nKekurangan:')
print('- Kehilangan interpretasi fitur asli')
print('- PCA assumes linear relationships')
print('- Sensitive to scale of features')

In [ ]:
# Buat label kualitas otomatis berdasarkan fitur
print('='*70)
print('CREATING QUALITY LABELS')
print('='*70)

def hitung_skor_kualitas(row):
    """
    Hitung skor kualitas (0-100) berdasarkan fitur
    """
    skor = 0
    
    # Sharpness (0-25 poin)
    if row['sharpness'] > 100:
        skor += 25
    elif row['sharpness'] > 50:
        skor += 15
    elif row['sharpness'] > 10:
        skor += 5
    
    # OCR Confidence (0-30 poin)
    if row['ocr_confidence'] > 80:
        skor += 30
    elif row['ocr_confidence'] > 60:
        skor += 20
    elif row['ocr_confidence'] > 40:
        skor += 10
    
    # Brightness optimal sekitar 128 (0-15 poin)
    brightness = row['mean_brightness']
    if 100 <= brightness <= 160:
        skor += 15
    elif 80 <= brightness <= 180:
        skor += 10
    elif 60 <= brightness <= 200:
        skor += 5
    
    # Contrast (0-15 poin)
    if row['contrast'] > 50:
        skor += 15
    elif row['contrast'] > 30:
        skor += 10
    elif row['contrast'] > 15:
        skor += 5
    
    # Edge Density (0-15 poin)
    if row['edge_density'] > 0.1:
        skor += 15
    elif row['edge_density'] > 0.05:
        skor += 10
    elif row['edge_density'] > 0.02:
        skor += 5
    
    return min(skor, 100)

# Apply ke dataframe
df['quality_score'] = df.apply(hitung_skor_kualitas, axis=1)

# Buat kategori label
def kategori_kualitas(skor):
    if skor >= 70:
        return 2  # Bagus
    elif skor >= 40:
        return 1  # Sedang
    else:
        return 0  # Buruk

df['quality_label'] = df['quality_score'].apply(kategori_kualitas)
df['quality_category'] = df['quality_label'].map({0: 'Buruk', 1: 'Sedang', 2: 'Bagus'})

print('\nLabel berhasil dibuat!')
print('\nDistribusi Kategori Kualitas:')
print(df['quality_category'].value_counts())

print('\nStatistik Quality Score:')
print(df['quality_score'].describe())

# Visualisasi distribusi kualitas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram quality score
axes[0].hist(df['quality_score'], bins=15, color='steelblue', 
            edgecolor='black', alpha=0.7)
axes[0].axvline(df['quality_score'].mean(), color='red', 
            linestyle='--', linewidth=2, label=f'Mean: {df["quality_score"].mean():.1f}')
axes[0].set_title('Distribusi Quality Score', fontweight='bold')
axes[0].set_xlabel('Quality Score (0-100)')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bar chart kategori
kategori_counts = df['quality_category'].value_counts()
colors = ['lightcoral', 'gold', 'lightgreen']
axes[1].bar(kategori_counts.index, kategori_counts.values, 
         color=colors, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribusi Kategori Kualitas', fontweight='bold')
axes[1].set_ylabel('Jumlah Gambar')
for i, v in enumerate(kategori_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'quality_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. STATISTICAL ANALYSIS

Melakukan uji statistik parametrik dan non-parametrik.

In [ ]:
# === UJI PARAMETRIK 1: PEARSON CORRELATION ===
print('='*70)
print('UJI PARAMETRIK 1: PEARSON CORRELATION TEST')
print('='*70)

print('\nHipotesis:')
print('H0: Tidak ada korelasi signifikan antara sharpness dan OCR confidence')
print('H1: Ada korelasi signifikan antara sharpness dan OCR confidence')

# Pilih dua variabel buat diuji
var1 = df['sharpness']
var2 = df['ocr_confidence']

# Cek normalitas dulu (syarat Pearson)
stat1, p1 = normaltest(var1)
stat2, p2 = normaltest(var2)

print('\n' + '-'*70)
print('Uji Normalitas (Syarat Pearson Correlation):')
print('-'*70)
print(f'Sharpness - p-value: {p1:.4f}', end='')
if p1 > 0.05:
    print(' -> Distribusi NORMAL')
else:
    print(' -> Distribusi TIDAK NORMAL')

print(f'OCR Confidence - p-value: {p2:.4f}', end='')
if p2 > 0.05:
    print(' -> Distribusi NORMAL')
else:
    print(' -> Distribusi TIDAK NORMAL')

# Pearson Correlation Test
corr_coef, p_value = stats.pearsonr(var1, var2)

print('\n' + '-'*70)
print('Hasil Uji Pearson Correlation:')
print('-'*70)
print(f'Koefisien Korelasi: {corr_coef:.4f}')
print(f'p-value: {p_value:.4f}')

# Hitung t-statistic dan confidence interval
n = len(var1)
dfree = n - 2
t_stat = corr_coef * np.sqrt(dfree / (1 - corr_coef**2))

# Confidence interval 95%
se = np.sqrt((1 - corr_coef**2) / dfree)
ci_low = corr_coef - 1.96 * se
ci_high = corr_coef + 1.96 * se

# Effect size (r-squared)
r_squared = corr_coef ** 2

print(f't-statistic: {t_stat:.4f}')
print(f'degrees of freedom: {dfree}')
print(f'95% CI: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'Effect Size (R²): {r_squared:.4f}')

print('\n' + '='*70)
print('INTERPRETASI UJI PEARSON CORRELATION')
print('='*70)

print('\n1. Nilai p-value:')
if p_value < 0.05:
    print(f'   p-value ({p_value:.4f}) < 0.05 -> MENOLAK H0')
    print('   Kesimpulan: Ada korelasi SIGNIFIKAN antara sharpness dan OCR confidence')
else:
    print(f'   p-value ({p_value:.4f}) >= 0.05 -> GAGAL MENOLAK H0')
    print('   Kesimpulan: Tidak ada korelasi signifikan')

print('\n2. Kekuatan Korelasi (berdasarkan koefisien):')
if abs(corr_coef) < 0.3:
    strength = 'LEMAR'
elif abs(corr_coef) < 0.7:
    strength = 'SEDANG'
else:
    strength = 'KUAT'

direction = 'POSITIF' if corr_coef > 0 else 'NEGATIF'
print(f'   Kekuatan: {strength}')
print(f'   Arah: {direction}')

print('\n3. Effect Size (R²):')
print(f'   {r_squared*100:.1f}% variasi OCR confidence dapat dijelaskan oleh sharpness')
if r_squared < 0.1:
    print('   Effect size KECIL')
elif r_squared < 0.3:
    print('   Effect size SEDANG')
else:
    print('   Effect size BESAR')

print('\n4. Confidence Interval:')
print(f'   95% CI: [{ci_low:.4f}, {ci_high:.4f}]')
if 0 < ci_low < 1 or 0 < ci_high < 1:
    print('   CI tidak termasuk 0 -> Korelasi SIGNIFIKAN')
else:
    print('   CI termasuk 0 -> Korelasi TIDAK SIGNIFIKAN')

In [ ]:
# === UJI PARAMETRIK 2: ONE-WAY ANOVA ===
print('\n' + '='*70)
print('UJI PARAMETRIK 2: ONE-WAY ANOVA')
print('='*70)

print('\nHipotesis:')
print('H0: Rata-rata quality_score SAMA di semua kategori kualitas')
print('H1: Minimal satu kategori punya rata-rata yang BERBEDA')

# Pisahkan data per kategori
groups_buruk = df[df['quality_category'] == 'Buruk']['quality_score']
groups_sedang = df[df['quality_category'] == 'Sedang']['quality_score']
groups_bagus = df[df['quality_category'] == 'Bagus']['quality_score']

# One-way ANOVA
f_stat, p_value = stats.f_oneway(groups_buruk, groups_sedang, groups_bagus)

print('\n' + '-'*70)
print('Hasil Uji ANOVA:')
print('-'*70)
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value: {p_value:.4f}')

# Effect size (Eta-squared)
grand_mean = df['quality_score'].mean()
ss_between = len(groups_buruk) * (groups_buruk.mean() - grand_mean)**2 + \
              len(groups_sedang) * (groups_sedang.mean() - grand_mean)**2 + \
              len(groups_bagus) * (groups_bagus.mean() - grand_mean)**2
ss_within = ((groups_buruk - groups_buruk.mean())**2).sum() + \
              ((groups_sedang - groups_sedang.mean())**2).sum() + \
              ((groups_bagus - groups_bagus.mean())**2).sum()
ss_total = ss_between + ss_within
eta_squared = ss_between / ss_total

print(f'Eta-squared (Effect size): {eta_squared:.4f}')

# Post-hoc test (Tukey HSD)
from scipy.stats import tukey_hsd

# Combine data dengan labels
all_data = list(groups_buruk) + list(groups_sedang) + list(groups_bagus)
all_labels = (['Buruk']*len(groups_buruk) + ['Sedang']*len(groups_sedang) + \
              ['Bagus']*len(groups_bagus))

tukey = tukey_hsd(all_data, all_labels)

print('\n' + '='*70)
print('INTERPRETASI UJI ANOVA')
print('='*70)

print('\n1. Nilai p-value:')
if p_value < 0.05:
    print(f'   p-value ({p_value:.4f}) < 0.05 -> MENOLAK H0')
    print('   Kesimpulan: Ada perbedaan SIGNIFIKAN rata-rata antar kategori')
else:
    print(f'   p-value ({p_value:.4f}) >= 0.05 -> GAGAL MENOLAK H0')
    print('   Kesimpulan: Tidak ada perbedaan signifikan rata-rata')

print('\n2. F-statistic:')
print(f'   F = {f_stat:.4f}')
print(f'   Interpretasi: Rata-rata antar grup berbeda sebesar {f_stat:.1f}x variansi dalam grup')

print('\n3. Effect Size (Eta-squared):')
print(f'   η² = {eta_squared:.4f}')
if eta_squared < 0.01:
    print('   Effect size KECIL (< 1%)')
elif eta_squared < 0.06:
    print('   Effect size SEDANG (1% - 6%)')
elif eta_squared < 0.14:
    print('   Effect size BESAR (6% - 14%)')
else:
    print('   Effect size SANGAT BESAR (> 14%)')

print('\n4. Mean per Kategori:')
print(f'   Buruk: {groups_buruk.mean():.2f}')
print(f'   Sedang: {groups_sedang.mean():.2f}')
print(f'   Bagus: {groups_bagus.mean():.2f}')

In [ ]:
# === UJI NON-PARAMETRIK 1: SPEARMAN CORRELATION ===
print('\n' + '='*70)
print('UJI NON-PARAMETRIK 1: SPEARMAN RANK CORRELATION')
print('='*70)

print('\nHipotesis:')
print('H0: Tidak ada korelasi monotonic antara edge_density dan quality_score')
print('H1: Ada korelasi monotonic antara edge_density dan quality_score')

# Pilih dua variabel
var1_spearman = df['edge_density']
var2_spearman = df['quality_score']

# Spearman Correlation Test
corr_spearman, p_spearman = spearmanr(var1_spearman, var2_spearman)

print('\n' + '-'*70)
print('Hasil Uji Spearman Correlation:')
print('-'*70)
print(f'Koefisien Korelasi (ρ): {corr_spearman:.4f}')
print(f'p-value: {p_spearman:.4f}')

print('\n' + '='*70)
print('INTERPRETASI UJI SPEARMAN')
print('='*70)

print('\n1. Nilai p-value:')
if p_spearman < 0.05:
    print(f'   p-value ({p_spearman:.4f}) < 0.05 -> MENOLAK H0')
    print('   Kesimpulan: Ada korelasi signifikan')
else:
    print(f'   p-value ({p_spearman:.4f}) >= 0.05 -> GAGAL MENOLAK H0')
    print('   Kesimpulan: Tidak ada korelasi signifikan')

print('\n2. Kekuatan Korelasi:')
if abs(corr_spearman) < 0.3:
    strength = 'LEMAR'
elif abs(corr_spearman) < 0.7:
    strength = 'SEDANG'
else:
    strength = 'KUAT'

direction = 'POSITIF' if corr_spearman > 0 else 'NEGATIF'
print(f'   Kekuatan: {strength}')
print(f'   Arah: {direction}')

print('\n3. Perbandingan dengan Pearson:')
print(f'   Spearman: {corr_spearman:.4f} (berdasarkan rank)')
print(f'   Pearson:  {corr_coef:.4f} (berdasarkan nilai asli)')
print('   Interpretasi: Kalau beda signifikan, ada outlier/nonlinear relationship')

In [ ]:
# === UJI NON-PARAMETRIK 2: KRUSKAL-WALLIS TEST ===
print('\n' + '='*70)
print('UJI NON-PARAMETRIK 2: KRUSKAL-WALLIS TEST')
print('='*70)

print('\nHipotesis:')
print('H0: Distribusi sharpness SAMA di semua kategori kualitas')
print('H1: Minimal satu kategori punya distribusi yang BERBEDA')

# Persiapkan data
groups_buruk_kw = df[df['quality_category'] == 'Buruk']['sharpness']
groups_sedang_kw = df[df['quality_category'] == 'Sedang']['sharpness']
groups_bagus_kw = df[df['quality_category'] == 'Bagus']['sharpness']

# Kruskal-Wallis Test
h_stat, p_kw = kruskal(groups_buruk_kw, groups_sedang_kw, groups_bagus_kw)

print('\n' + '-'*70)
print('Hasil Uji Kruskal-Wallis:')
print('-'*70)
print(f'H-statistic: {h_stat:.4f}')
print(f'p-value: {p_kw:.4f}')

# Effect size (Epsilon-squared approximation)
n_total = len(df)
epsilon_squared = (h_stat - len(df['quality_category'].unique()) + 1) / (n_total - 1)

print(f'Epsilon-squared: {epsilon_squared:.4f}')

print('\n' + '='*70)
print('INTERPRETASI UJI KRUSKAL-WALLIS')
print('='*70)

print('\n1. Nilai p-value:')
if p_kw < 0.05:
    print(f'   p-value ({p_kw:.4f}) < 0.05 -> MENOLAK H0')
    print('   Kesimpulan: Distribusi sharpness BERBEDA SIGNIFIKAN antar kategori')
else:
    print(f'   p-value ({p_kw:.4f}) >= 0.05 -> GAGAL MENOLAK H0')
    print('   Kesimpulan: Distribusi sharpness TIDAK berbeda signifikan')

print('\n2. H-statistic:')
print(f'   H = {h_stat:.4f}')
print(f'   Interpretasi: Perbedaan rank antar grup sebesar {h_stat:.1f}x')

print('\n3. Median per Kategori:')
print(f'   Buruk: {groups_buruk_kw.median():.2f}')
print(f'   Sedang: {groups_sedang_kw.median():.2f}')
print(f'   Bagus: {groups_bagus_kw.median():.2f}')

print('\n4. Keputusan:')
print('   Karena data tidak terlalu banyak, Kruskal-Wallis lebih cocok')
print('   dibanding ANOVA kalau asumsi normalitas terpenuhi')

In [ ]:
# Siapkan fitur dan target
print('='*70)
print('PERSIAPAN DATA TRAINING')
print('='*70)

# Pilih fitur yang akan dipakai
fitur_model = ['mean_brightness', 'std_brightness', 'sharpness', 'contrast',
              'edge_density', 'avg_gradient', 'texture_variance', 'noise_level',
              'entropy', 'hist_skew', 'hist_kurtosis', 'aspect_ratio',
              'ocr_confidence', 'text_length', 'word_count']

X = df[fitur_model].values
y_regression = df['quality_score'].values
y_classification = df['quality_label'].values

# Split data: 80% train, 20% test
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_regression, y_classification, test_size=0.2, random_state=42
)

print(f'\nData Train: {X_train.shape[0]} sampel')
print(f'Data Test: {X_test.shape[0]} sampel')
print(f'Fitur: {X_train.shape[1]} fitur')

# Scaling fitur
scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train)
X_test_scaled = scaler_ml.transform(X_test)

print('\nFitur yang dipakai:')
print('-'*70)
for i, fitur in enumerate(fitur_model, 1):
    print(f'{i:2d}. {fitur}')
print(f'\nTotal: {len(fitur_model)} fitur')

In [ ]:
# === TRAINING MODEL ===
print('\n' + '='*70)
print('TRAINING MACHINE LEARNING MODEL')
print('='*70)

# Model 1: Random Forest Regression (Prediksi Quality Score 0-100)
print('\n1. Random Forest Regression...')
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train_scaled, y_train_reg)

rf_reg_pred = rf_reg.predict(X_test_scaled)
rf_reg_mae = mean_absolute_error(y_test_reg, rf_reg_pred)
rf_reg_r2 = r2_score(y_test_reg, rf_reg_pred)

print(f'   MAE: {rf_reg_mae:.4f}')
print(f'   R² Score: {rf_reg_r2:.4f}')

# Model 2: Random Forest Classification (Prediksi Kategori: Buruk/Sedang/Bagus)
print('\n2. Random Forest Classification...')
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_scaled, y_train_clf)

rf_clf_pred = rf_clf.predict(X_test_scaled)
rf_clf_acc = accuracy_score(y_test_clf, rf_clf_pred)

print(f'   Accuracy: {rf_clf_acc:.4f}')

# Model 3: Logistic Regression (Baseline comparison)
print('\n3. Logistic Regression...')
lr_clf = LogisticRegression(random_state=42, max_iter=1000)
lr_clf.fit(X_train_scaled, y_train_clf)

lr_clf_pred = lr_clf.predict(X_test_scaled)
lr_clf_acc = accuracy_score(y_test_clf, lr_clf_pred)

print(f'   Accuracy: {lr_clf_acc:.4f}')

print('\n' + '='*70)
print('PERBANDINGAN MODEL')
print('='*70)

df_perbandingan = pd.DataFrame({
    'Model': ['RF Regression', 'RF Classification', 'Logistic Regression'],
    'Task': ['Regression', 'Classification', 'Classification'],
    'Metric': ['R² Score', 'Accuracy', 'Accuracy'],
    'Score': [rf_reg_r2, rf_clf_acc, lr_clf_acc]
})

print(df_perbandingan.to_string(index=False))

print('\n' + '-'*70)
print('Kesimpulan:')
print('-'*70)

if rf_clf_acc > lr_clf_acc:
    print('Random Forest lebih akurat dibanding Logistic Regression')
    print(f'Selisih akurasi: {(rf_clf_acc - lr_clf_acc)*100:.2f}%')
else:
    print('Logistic Regression lebih akurat dibanding Random Forest')
    print(f'Selisih akurasi: {(lr_clf_acc - rf_clf_acc)*100:.2f}%')

In [ ]:
# Feature Importance dari Random Forest
print('\n' + '='*70)
print('FEATURE IMPORTANCE - RANDOM FOREST')
print('='*70)

# Feature importance dari classifier
feature_importance = pd.DataFrame({
    'Feature': fitur_model,
    'Importance': rf_clf.feature_importances_
})
feature_importance = feature_importance.sort_values('Importance', ascending=False)

print('\nTop 10 Fitur Terpenting:')
print(feature_importance.head(10).to_string(index=False))

# Visualisasi Feature Importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'][:10], feature_importance['Importance'][:10],
         color='steelblue', edgecolor='black')
plt.xlabel('Importance Score', fontweight='bold')
plt.title('Top 10 Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretasi:')
print('- Fitur dengan importance tinggi paling berpengaruh ke kualitas')
print('- Fitur dengan importance rendah bisa dipertimbangkan untuk dihapus')

In [ ]:
# Classification Report
from sklearn.metrics import confusion_matrix, classification_report

print('\n' + '='*70)
print('CLASSIFICATION REPORT - RANDOM FOREST')
print('='*70)

print(classification_report(y_test_clf, rf_clf_pred, 
                          target_names=['Buruk', 'Sedang', 'Bagus']))

# Confusion Matrix
cm = confusion_matrix(y_test_clf, rf_clf_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
          xticklabels=['Buruk', 'Sedang', 'Bagus'],
          yticklabels=['Buruk', 'Sedang', 'Bagus'])
plt.title('Confusion Matrix - Random Forest', fontweight='bold', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretasi Confusion Matrix:')
for i in range(len(cm)):
    for j in range(len(cm[i])):
        if cm[i,j] > 0:
            label_actual = ['Buruk', 'Sedang', 'Bagus'][i]
            label_pred = ['Buruk', 'Sedang', 'Bagus'][j]
            print(f'{label_actual} -> {label_pred}: {cm[i,j]} gambar')

In [ ]:
# === VISUALISASI PERBANDINGAN MODEL ===
# Visualisasi actual vs predicted untuk regression
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot: Actual vs Predicted (Regression)
axes[0].scatter(y_test_reg, rf_reg_pred, alpha=0.6, color='blue', edgecolors='black', linewidth=0.5)
axes[0].plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Quality Score', fontweight='bold')
axes[0].set_ylabel('Predicted Quality Score', fontweight='bold')
axes[0].set_title(f'Random Forest Regression\nR²={rf_reg_r2:.4f}, MAE={rf_reg_mae:.2f}', 
              fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bar chart: Accuracy Comparison
models = ['Random Forest\n(Classification)', 'Logistic\nRegression']
accuracies = [rf_clf_acc, lr_clf_acc]
colors = ['green' if acc == max(accuracies) else 'gray' for acc in accuracies]

bars = axes[1].bar(models, accuracies, color=colors, edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Accuracy', fontweight='bold')
axes[1].set_title('Model Accuracy Comparison', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accuracies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{acc:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*70)
print('INTERPRETASI VISUALISASI MODEL')
print('='*70)

print('\n1. Scatter Plot (Actual vs Predicted):')
print(f'   R² Score: {rf_reg_r2:.4f}')
if rf_reg_r2 > 0.7:
    print('   Model SANGAT BAGUS - bisa memprediksi quality score dengan akurat')
elif rf_reg_r2 > 0.5:
    print('   Model BAGUS - cukup akurat buat prediksi')
else:
    print('   Model perlu improvement')

print('\n2. Accuracy Comparison:')
print(f'   Random Forest: {rf_clf_acc:.4f}')
print(f'   Logistic Regression: {lr_clf_acc:.4f}')
print(f'   Selisih: {abs(rf_clf_acc - lr_clf_acc):.4f}')
print('\n   Random Forest lebih unggul untuk klasifikasi kualitas gambar')

In [ ]:
# Simpan model dan metadata
print('='*70)
print('MENYIMPAN MODEL YANG SUDAH DILATIH')
print('='*70)

# Buat folder untuk menyimpan model
save_dir = os.path.join(MODEL_DIR, 'saved_models')
os.makedirs(save_dir, exist_ok=True)

# Simpan model-model
joblib.dump(rf_reg, os.path.join(save_dir, 'rf_regressor.pkl'))
joblib.dump(rf_clf, os.path.join(save_dir, 'rf_classifier.pkl'))
joblib.dump(lr_clf, os.path.join(save_dir, 'lr_classifier.pkl'))
joblib.dump(scaler_ml, os.path.join(save_dir, 'scaler.pkl'))

# Simpan metadata
metadata = {
    'project': 'OCR Quality Check',
    'trained_at': datetime.now().isoformat(),
    'feature_cols': fitur_model,
    'n_features': len(fitur_model),
    'n_samples': len(df),
    'n_train': len(X_train),
    'n_test': len(X_test),
    'models': {
        'rf_regressor': {
            'mae': float(rf_reg_mae),
            'r2_score': float(rf_reg_r2)
        },
        'rf_classifier': {
            'accuracy': float(rf_clf_acc)
        },
        'lr_classifier': {
            'accuracy': float(lr_clf_acc)
        }
    },
    'preprocessing': {
        'scaling': 'StandardScaler',
        'outlier_handling': 'Winsorization (5th-95th percentile)',
        'missing_values': 'Mean imputation'
    },
    'statistical_tests': {
        'pearson_correlation': {
            'variables': ['sharpness', 'ocr_confidence'],
            'coefficient': float(corr_coef),
            'p_value': float(p_value),
            'significant': p_value < 0.05
        },
        'anova': {
            'f_statistic': float(f_stat),
            'p_value': float(p_value),
            'eta_squared': float(eta_squared)
        },
        'spearman_correlation': {
            'variables': ['edge_density', 'quality_score'],
            'coefficient': float(corr_spearman),
            'p_value': float(p_spearman),
            'significant': p_spearman < 0.05
        },
        'kruskal_wallis': {
            'h_statistic': float(h_stat),
            'p_value': float(p_kw),
            'epsilon_squared': float(epsilon_squared)
        }
    }
}

metadata_path = os.path.join(save_dir, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('\nModel berhasil disimpan!')
print(f'Folder: {save_dir}')
print('\nFile yang disimpan:')
print('  - rf_regressor.pkl')
print('  - rf_classifier.pkl')
print('  - lr_classifier.pkl')
print('  - scaler.pkl')
print('  - metadata.json')

In [ ]:
# Fungsi prediksi kualitas gambar baru
def prediksi_kualitas_gambar(image_path, model, scaler, feature_cols):
    """
    Prediksi kualitas gambar baru
    
    Args:
        image_path: Path ke gambar
        model: Model yang sudah dilatih
        scaler: Scaler yang sudah fit
        feature_cols: Daftar nama fitur
    
    Returns:
        dict: Hasil prediksi
    """
    try:
        # Extract fitur
        fitur, img, gray = extract_fitur_gambar(image_path)
        
        if fitur is None:
            return {'error': 'Gagal memuat gambar'}
        
        # Siapkan fitur
        X_new = np.array([[fitur[col] for col in feature_cols]])
        
        # Scale
        X_new_scaled = scaler.transform(X_new)
        
        # Prediksi skor
        skor = model.predict(X_new_scaled)[0]
        
        # Prediksi kategori
        label_idx = model.predict(X_new_scaled)[0]
        label = ['Buruk', 'Sedang', 'Bagus'][label_idx]
        
        # Probability
        probs = model.predict_proba(X_new_scaled)[0]
        
        return {
            'image_path': image_path,
            'quality_score': round(skor, 2),
            'quality_category': label,
            'probabilities': {
                'Buruk': round(probs[0], 4),
                'Sedang': round(probs[1], 4),
                'Bagus': round(probs[2], 4)
            }
        }
        
    except Exception as e:
        return {'error': str(e)}

print('Fungsi prediksi siap!')

In [ ]:
# Test dengan beberapa gambar dari dataset
print('='*70)
print('TEST MODEL DENGAN GAMBAR BARU')
print('='*70)

# Ambil beberapa gambar random buat test
test_files = list(Path(TRAIN_DIR).glob('*.png'))[:5]

for img_path in test_files:
    print(f'\nGambar: {img_path.name}')
    print('-'*50)
    
    result = prediksi_kualitas_gambar(str(img_path), rf_clf, scaler_ml, fitur_model)
    
    if 'error' in result:
        print(f'Error: {result["error"]}')
        continue
    
    print(f'Quality Score: {result["quality_score"]}/100')
    print(f'Kategori: {result["quality_category"]}')
    print(f'\nProbabilitas:')
    for label, prob in result['probabilities'].items():
        print(f'  {label}: {prob*100:.2f}%')
    
    # Tampilkan gambar
    img = cv2.imread(str(img_path))
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f'{img_path.name} | {result["quality_category"]} | Score: {result["quality_score"]}', 
                 fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()